# Tutorial for PUBMED queries + Claude 

Before we start with the code in this jupyter notebook, we first need to setup the Zotero API key and Anthropic API key in your enviroment

# Loading packages

First, we load the relevant packages

In [1]:
import argparse  # Parse command-line arguments to configure pipeline execution
import json  # Read and write structured data in JSON format
import os  # Interact with the operating system (files, paths, environment)
import xml.etree.ElementTree as ET  # Parse and navigate PubMed XML responses
from collections import defaultdict  # Dictionary with automatic default values
from datetime import datetime, timedelta  # Handle dates and date arithmetic
from time import sleep  # Pause execution to respect API rate limits
import anthropic  # Client library for interacting with Claude LLM API
import requests  # Send HTTP requests to PubMed and other REST APIs
from diskcache import Cache  # Persistent on-disk caching to avoid repeated API calls
from pyzotero import Zotero  # Interface with Zotero API for reference management

In [2]:
cache = Cache(".cache")  # Initialize persistent on-disk cache for API results and LLM outputs

zot = None  # Default to no Zotero connection unless credentials are provided
if "ZOTERO_API_KEY" in os.environ:
    zot = Zotero(
        "6377183", "group", os.environ["ZOTERO_API_KEY"]
    )  # Connect to Zotero group library using API key (cloud-based access)

# Initialize Claude client
api_key = os.environ.get("ANTHROPIC_API_KEY")  # Read Claude API key from environment variables

anthropic_client = None  # Default to no LLM client if API key is missing
if "ANTHROPIC_API_KEY" in os.environ:
    anthropic_client = anthropic.Anthropic(
        api_key=os.environ["ANTHROPIC_API_KEY"]
    )  # Create Claude client for paper classification and summarization

BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"  # Base URL for NCBI PubMed E-utilities API

# Search terms / Categories

In [3]:
MAIN_QUERY = """(
  "machine learning"[MeSH Terms]
  OR "artificial intelligence"[MeSH Terms]
  OR "machine learning"[Title/Abstract]
  OR "artificial intelligence"[Title/Abstract]
  OR "deep learning"[Title/Abstract]
  OR "neural network*"[Title/Abstract]
  OR "random forest"[Title/Abstract]
  OR "support vector machine*"[Title/Abstract]
  OR "Gaussian process*"[Title/Abstract]
  OR "reinforcement learning"[Title/Abstract]
  OR "Bayesian machine learning"[Title/Abstract]
  OR "hybrid model*"[Title/Abstract]
  OR "mechanism-informed"[Title/Abstract]
)
AND
(
  pharmacokinetic*[Title/Abstract]
  OR pharmacodynamic*[Title/Abstract]
  OR PK/PD[Title/Abstract]
  OR "population pharmacokinetic*"[Title/Abstract]
  OR "nonlinear mixed effects"[Title/Abstract]
  OR NLME[Title/Abstract]
  OR PBPK[Title/Abstract]
  OR "physiologically based pharmacokinetic*"[Title/Abstract]
  OR QSP[Title/Abstract]
  OR "quantitative systems pharmacology"[Title/Abstract]
  OR "model-based drug development"[Title/Abstract]
  OR "model-informed drug development"[Title/Abstract]
  OR MIDD[Title/Abstract]
  OR pharmacometrics[Title/Abstract]
  OR "precision dosing"[Title/Abstract]
  OR "dose optimization"[Title/Abstract]
  OR "therapeutic drug monitoring"[Title/Abstract]
)
"""

FALLBACK_TAG = "Other/General"

# Updated mini-list of pharmacometrics applications
PMX_APPLICATION_TAGS = [
    "Outcome prediction",
    "Covariate selection / confounding adjustment",
    "Pharmacometric modeling (Pharmacokinetic modeling, survival analysis, exposure–response analysis, pharmacodynamic modeling)",
    "RWD phenotyping",
    "Drug toxicity prediction",
    "Drug repurposing",
    "Enrichment design",
    "Patient risk stratification / management",
    "Dose selection / optimization",
    "Adherence to drug regimen",
    "Synthetic control",
    "Postmarketing surveillance",
    "Endpoint / biomarker assessment",
    "Disease progression modeling",
    "Automation of PK/PD modeling",
    "Precision medicine / optimized treatment regimen",
    "Causal inference",
    "Data imputation",
    "Discovery of subpatient groups"
]

PAPER_TYPE_TAGS = ["review", "tutorial", "perspective"]

METHODOLOGY_TAGS = [
    "Supervised learning",
    "Unsupervised learning",
    "Deep learning",
    "Tree-based models",
    "Gaussian processes",
    "Bayesian ML",
    "Hybrid mechanistic–ML models",
    "Feature selection",
    "Model selection",
    "Surrogate modeling",
    "Emulation of NLME models",
    "Reinforcement learning",
    "Explainable AI",
    "Neural networks",
    "Ensemble learning",
    "Time-series modeling",
    "Mechanism-informed machine learning",
    "LLM",
    "AI Agents"
]

# Helper Functions

Let's go through each function one by one:

## ***get_pmids***: querying PubMed for recent papers

In [4]:
def get_pmids(
    base_query: str,
    days_back: int = 1,
    max_results: int = 200,
):
    """
    Query PubMed using a fixed Boolean query plus a sliding publication date window.

    Parameters
    ----------
    base_query : str
        PubMed Boolean query (e.g. pharmacometrics OR clinical pharmacology)
    days_back : int
        How many days back from today to search
    max_results : int
        Maximum number of PMIDs to return

    Returns
    -------
    list[str]
        List of PubMed IDs (PMIDs)
    """

    # Get today's date in UTC (PubMed uses publication dates, not local time)
    end_date = datetime.utcnow().date()

    # Compute the start date by subtracting days_back
    start_date = end_date - timedelta(days=days_back)

    # Construct PubMed date filter syntax
    # Example:
    # "2025/01/30"[Date - Publication] : "2025/01/31"[Date - Publication]
    date_clause = (
        f'"{start_date.strftime("%Y/%m/%d")}"[Date - Publication] : '
        f'"{end_date.strftime("%Y/%m/%d")}"[Date - Publication]'
    )

    # Combine the base query with the date constraint
    # Using parentheses ensures correct Boolean precedence
    full_query = f"""
    ({base_query})
    AND
    ({date_clause})
    """

    # PubMed E-utilities endpoint for searching
    search_url = f"{BASE_URL}esearch.fcgi"

    # Parameters passed to PubMed
    search_params = {
        "db": "pubmed",        # database to search
        "term": full_query,    # search query
        "retmax": max_results, # max number of results
        "retmode": "json",     # JSON response (easier to parse)
        "usehistory": "n",     # do not store query on NCBI servers
    }

    # Execute HTTP GET request with a timeout for safety
    response = requests.get(search_url, params=search_params, timeout=30)

    # Raise an exception if PubMed returns HTTP errors (4xx / 5xx)
    response.raise_for_status()

    # Parse JSON response into Python dict
    data = response.json()

    # Safely extract list of PMIDs
    # If any key is missing, return an empty list
    return data.get("esearchresult", {}).get("idlist", [])

## ***get_article_entry***: helper to extract XML text safely

In [5]:
def get_article_entry(elem, key):
    """
    Extract text from an XML element using XPath-like syntax.

    Parameters
    ----------
    elem : xml.etree.ElementTree.Element
        Parent XML element (e.g. PubmedArticle)
    key : str
        Tag or XPath to search for

    Returns
    -------
    str or None
        Extracted text, or None if not found
    """

    # Find the first matching element
    res = elem.find(f".//{key}")

    # If found, return all text inside (handles nested tags)
    if res is not None:
        return "".join(res.itertext())

## ***query_pmid***: fetch and normalize one PubMed article

In [6]:
@cache.memoize()
def query_pmid(pmid):
    """
    Fetch full metadata for a single PubMed ID and convert it
    into a Zotero-compatible dictionary.

    Cached to avoid repeated API calls.
    """

    # PubMed efetch endpoint retrieves full records
    fetch_url = f"{BASE_URL}efetch.fcgi"

    # Parameters: database, PMID, XML output
    fetch_params = {"db": "pubmed", "id": pmid, "retmode": "xml"}

    # HTTP request to PubMed
    fetch_response = requests.get(fetch_url, params=fetch_params)

    # Parse XML response
    root = ET.fromstring(fetch_response.content)

    articles = []

    # Loop over PubmedArticle elements (normally exactly one per PMID)
    for article in root.findall(".//PubmedArticle"):

        # Extract PMID again from XML (safety)
        pmid = get_article_entry(article, "PMID")

        # Build a structured article dictionary
        article_dict = {
            "itemType": "journalArticle",  # Zotero item type
            "title": get_article_entry(article, "ArticleTitle"),
            "abstractNote": get_article_entry(article, "AbstractText"),
            "PMID": pmid,
            "date": get_article_entry(article, "PubDate"),
            "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
            "DOI": get_article_entry(article, "ArticleId[@IdType='doi']"),

            # Build list of authors
            "creators": [
                {
                    "creatorType": "author",
                    "firstName": author.findtext("ForeName"),
                    "lastName": author.findtext("LastName"),
                }
                for author in article.findall(".//AuthorList/Author")
            ],
        }

        articles.append(article_dict)

    # Sanity check: PubMed should not return multiple articles per PMID
    if len(articles) > 1:
        raise RuntimeError(f"Too many articles for given PMID ({pmid})")

    return articles[0]

## ***classify_paper***: LLM-based classification with Claude

In [38]:
# Optional caching
# @cache.memoize()  
def classify_paper(title, abstract):
    """
    Classify a paper into multiple axes: paper_type, application, methodology.
    Always returns an AI summary.

    Rules:
    - If all axes are empty, assign [Other/General]
    - If at least one axis has a tag, only use the tags returned by Claude
    - Summary is always returned
    """

    if abstract is None:
        abstract = ""

    # --- Claude not configured ---
    if anthropic_client is None:
        return (
            {
                "paper_type": [FALLBACK_TAG],
                "application": [FALLBACK_TAG],
                "methodology": [FALLBACK_TAG],
            },
            "Claude not configured"
        )

    # Build the prompt dynamically from external tag lists
    prompt = f"""
You are an experienced pharmacometrician with expertise in AI/ML applications in drug development and clinical pharmacology.

Classify the following paper into ZERO OR MORE tags per category.
Only assign a tag if clearly supported by the title or abstract.
Return STRICT JSON only.

Paper Title:
{title}

Abstract:
{abstract[:1500]}

Tag schema:

paper_type (choose any):
{chr(10).join(f"- {t}" for t in PAPER_TYPE_TAGS)}

application (choose any):
{chr(10).join(f"- {t}" for t in PMX_APPLICATION_TAGS)}

methodology (choose any):
{chr(10).join(f"- {t}" for t in METHODOLOGY_TAGS)}

Response format:
{{
  "paper_type": [],
  "application": [],
  "methodology": [],
  "summary": "one sentence summary (max 150 chars)"
}}
"""

    try:
        # --- Call Claude ---
        message = anthropic_client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=500,
            temperature=0,
            messages=[{"role": "user", "content": prompt}],
        )

        response_text = message.content[0].text

        # --- Default summary fallback ---
        summary = "AI/ML application in pharmacometrics or clinical pharmacology"

        # Try to parse JSON from Claude response
        try:
            start_idx = response_text.find("{")
            end_idx = response_text.rfind("}") + 1
            result = json.loads(response_text[start_idx:end_idx])
            summary = result.get("summary", summary)

        except Exception:
            result = {}

        # --- Extract tags per axis ---
        classification = {}
        for axis in ["paper_type", "application", "methodology"]:
            tags = result.get(axis, []) if isinstance(result, dict) else []
            classification[axis] = tags

        # --- If all axes empty, assign fallback Other/General only to application ---
        if all(len(tags) == 0 for tags in classification.values()):
            classification["application"] = [FALLBACK_TAG]

        return classification, summary

    except Exception as e:
        print(f"Error classifying paper '{title}': {e}")
        return (
            {
                "paper_type": [FALLBACK_TAG],
                "application": [FALLBACK_TAG],
                "methodology": [FALLBACK_TAG],
            },
            "Error during classification"
        )

## Classify_and_prepare

In [49]:
# --- Pipeline to classify all articles and prepare outputs ---
def classify_and_prepare(articles, zot=None):
    """
    Classify papers, update articles dict, prepare cat_map for README, 
    and Zotero tags if zot is provided.
    """

    # Only store PMX applications as README sections
    cat_map = defaultdict(list)

    for pmid, article in articles.items():
        classification, summary = classify_paper(article["title"], article.get("abstractNote", ""))
        article["classification"] = classification
        article["extra"] = summary  # Always keep AI summary

        # --- README aggregation: only use PMX applications as section headers ---
        pmx_apps = classification.get("application", [])
        pmx_apps = [t for t in pmx_apps if t != FALLBACK_TAG]
        if not pmx_apps:
            pmx_apps = [FALLBACK_TAG]

        for app in pmx_apps:
            cat_map[app].append(pmid)

        # --- Prepare Zotero tags ---
        if zot is not None:
            zot_tags = []
            for axis, tags in classification.items():
                for t in tags:
                    zot_tags.append({"tag": f"{axis}:{t}"})
            article_for_zotero = article.copy()
            article_for_zotero["tags"] = zot_tags
            try:
                zot.create_items([article_for_zotero])
                sleep(1)
            except Exception as e:
                print(f"Error uploading {pmid} to Zotero: {e}")

    return cat_map

## Create TOC

In [66]:
def generate_readme_toc(cat_map):
    """
    Generate an alphabetical Markdown Table of Contents for PMX applications.
    """
    toc_lines = ["## Table of Contents\n"]
    for cat in sorted(cat_map.keys()):  # Sort alphabetically
        link = cat.lower().replace(" ", "-")
        toc_lines.append(f"- [{cat}](#{link})")
    return "\n".join(toc_lines) + "\n"


## ***update_readme***: generate human-readable output

In [67]:
def update_readme(articles, cat_map, filename="README.md"):
    header = f"""# Awesome AI/ML Applications in Pharmacometrics 🧬🤖

A curated list of research papers on AI/ML applications in pharmacometrics and clinical pharmacology, regularly updated.

**Last Updated**: {datetime.now().strftime('%Y-%m-%d')}

---
"""

    toc = generate_readme_toc(cat_map)

    with open(filename, "w") as fh:
        fh.write(header)
        fh.write(toc)

        review_pmids = []

        # Write PMX application sections in alphabetical order
        for cat in sorted(cat_map.keys()):
            pmids = cat_map[cat]
            fh.write(f"\n## {cat}\n")
            for pmid in pmids:
                article = articles[pmid]
                classification = article.get("classification", {})
                paper_type = classification.get("paper_type", [])

                # Collect reviews/tutorials/perspectives for bottom
                if any(pt in ["review", "tutorial", "perspective"] for pt in paper_type):
                    review_pmids.append(pmid)
                    continue

                methodology = classification.get("methodology", [])
                methodology_str = f"Methodology: {', '.join(methodology)}" if methodology else ""

                fh.write(
                    f"\n- **[{article['title']}]({article['url']})**\n"
                    f"\t- {methodology_str}\n"
                    f"\t- Published: {article.get('date', 'N/A')}\n"
                    f"\t- Summary: {article.get('extra', '')}\n"
                )

        # Append reviews/tutorials/perspectives at the bottom (also alphabetical by title)
        if review_pmids:
            fh.write("\n## Reviews / Tutorials / Perspectives\n")
            for pmid in sorted(review_pmids, key=lambda x: articles[x]['title']):
                article = articles[pmid]
                classification = article.get("classification", {})
                methodology = classification.get("methodology", [])
                methodology_str = f"Methodology: {', '.join(methodology)}" if methodology else ""

                fh.write(
                    f"\n- **[{article['title']}]({article['url']})**\n"
                    f"\t- {methodology_str}\n"
                    f"\t- Published: {article.get('date', 'N/A')}\n"
                    f"\t- Summary: {article.get('extra', '')}\n"
                )


## ***main***: orchestration layer

In [9]:
def main(
    filename="all_articles.json",
    readme_path="README.md",
    days_back=365,
    max_results=100,
):
    """
    Full pipeline:
    - Load existing articles
    - Query PubMed for recent papers
    - Classify using Claude
    - Prepare cat_map based on PMX applications
    - Upload to Zotero if configured
    - Update README
    """

    # --- Load previously fetched articles ---
    articles = {}
    if os.path.isfile(filename):
        with open(filename, "r") as fh:
            articles = json.load(fh)

    # --- Query PubMed ---
    print("🔬 Fetching recent AI/ML pharmacometrics papers from PubMed...")
    pmids = get_pmids(REVIEW_PUBMED_QUERY_PMX, days_back=days_back, max_results=max_results)
    print(f"Found {len(pmids)} PMIDs")

    # --- Fetch new articles ---
    new_pmids = [pmid for pmid in pmids if pmid not in articles]
    for pmid in new_pmids:
        try:
            articles[pmid] = query_pmid(pmid)
        except Exception as e:
            print(f"Error fetching PMID {pmid}: {e}")

    print(f"Total articles in memory: {len(articles)}")

    # --- Determine which PMIDs need upload to Zotero ---
    pmids_to_upload = set()
    if zot is not None:
        existing_pmids_in_zot = {x["data"]["PMID"] for x in zot.items()}
        pmids_to_upload = set(articles.keys()) - existing_pmids_in_zot

    # --- Classify and prepare articles ---
    print("🤖 Classifying papers with Claude and preparing Zotero...")
    cat_map = classify_and_prepare(articles, zot=zot)

    print("📝 Updating JSON file...")
    # --- Update JSON file ---
    with open(filename, "w") as fh:
        json.dump(articles, fh, indent=2)

    # --- Update README ---
    print("📝 Updating README.md...")
    update_readme(articles, cat_map, filename=readme_path)
    print("✅ Pipeline complete!")


In [58]:
filename="all_articles.json"
readme_path="../README.md"
days_back=365
max_results=30

In [47]:
# --- Load previously fetched articles ---
articles = {}
if os.path.isfile(filename):
    with open(filename, "r") as fh:
        articles = json.load(fh)

# --- Query PubMed ---
print("🔬 Fetching recent AI/ML pharmacometrics papers from PubMed...")
pmids = get_pmids(MAIN_QUERY, days_back=days_back, max_results=max_results)
print(f"Found {len(pmids)} PMIDs")

# --- Fetch new articles ---
new_pmids = [pmid for pmid in pmids if pmid not in articles]
for pmid in new_pmids:
    try:
        articles[pmid] = query_pmid(pmid)
    except Exception as e:
        print(f"Error fetching PMID {pmid}: {e}")

print(f"Total articles in memory: {len(articles)}")

🔬 Fetching recent AI/ML pharmacometrics papers from PubMed...


/tmp/ipykernel_30045/2233061368.py:25: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_date = datetime.utcnow().date()


Found 30 PMIDs
Total articles in memory: 30


In [50]:
# --- Determine which PMIDs need upload to Zotero ---
pmids_to_upload = set()
if zot is not None:
    existing_pmids_in_zot = {x["data"]["PMID"] for x in zot.items()}
    pmids_to_upload = set(articles.keys()) - existing_pmids_in_zot
# --- Classify and prepare articles ---
print("🤖 Classifying papers with Claude and preparing Zotero...")
cat_map = classify_and_prepare(articles, zot=zot)
print("📝 Updating JSON file...")


🤖 Classifying papers with Claude and preparing Zotero...
📝 Updating JSON file...


In [51]:
# --- Update JSON file ---
with open(filename, "w") as fh:
    json.dump(articles, fh, indent=2)

📝 Updating README.md...
✅ Pipeline complete!


In [68]:
# --- Update README ---
print("📝 Updating README.md...")
update_readme(articles, cat_map, filename=readme_path)
print("✅ Pipeline complete!")


📝 Updating README.md...
✅ Pipeline complete!


# Testing functions

In [ ]:
pmids = get_pmids(
    MAIN_QUERY,
    days_back=365*25,
    max_results=5000
)
len(pmids)

In [ ]:
for pmid in pmids[:30]:
    article = query_pmid(pmid)
    print(f"- {article['title']}")


# Testing the functions
 Different queries

In [ ]:
ML_BLOCK = """
(
  "machine learning"[tiab] OR
  "artificial intelligence"[tiab] OR
  "deep learning"[tiab] OR
  "neural network"[tiab] OR
  "random forest"[tiab] OR
  "gradient boosting"[tiab] OR
  "XGBoost"[tiab] OR
  "support vector"[tiab]
)
"""
PMX_BLOCK = """
(
  "pharmacometrics"[tiab] OR
  "population pharmacokinetic*"[tiab] OR
  "population pharmacodynamic*"[tiab] OR
  "nonlinear mixed effect*"[tiab] OR
  "mixed effect* model*"[tiab] OR
  "NLME"[tiab] OR
  "model-based drug development"[tiab] OR
  "MIDD"[tiab] OR
  "NONMEM"[tiab] OR
  "Monolix"[tiab] OR
  "nlmixr"[tiab] OR
  "Pumas"[tiab]
)
AND
(
  "clinical pharmacology"[tiab] OR
  "drug development"[tiab] OR
  "clinical trial"[tiab] OR
  "dose finding"[tiab] OR
  "dose optimization"[tiab] OR
  "exposure response"[tiab] OR
  "model-informed drug development"[tiab] OR
  pharmacokinetic*[tiab] OR
  pharmacodynamic*[tiab] OR
  pharmacometric*[tiab]
)
"""
EXCLUDE_BLOCK = """
NOT (
  review[Publication Type] OR
  "electronic health record"[tiab] OR
  "EHR"[tiab] OR
  "radiomics"[tiab] OR
  "genome-wide"[tiab]
)
"""
BASE_PUBMED_QUERY_PMX = f"""
{ML_BLOCK}
AND
{PMX_BLOCK}
{EXCLUDE_BLOCK}
"""



In [ ]:
BROAD_QUERY = """(
  "machine learning"[MeSH Terms]
  OR "artificial intelligence"[MeSH Terms]
  OR "machine learning"[Title/Abstract]
  OR "artificial intelligence"[Title/Abstract]
  OR "deep learning"[Title/Abstract]
  OR "neural network*"[Title/Abstract]
  OR "random forest"[Title/Abstract]
  OR "support vector machine*"[Title/Abstract]
  OR "Gaussian process*"[Title/Abstract]
  OR "reinforcement learning"[Title/Abstract]
)
AND
(
  pharmacokinetic*[Title/Abstract]
  OR pharmacodynamic*[Title/Abstract]
  OR PK/PD[Title/Abstract]
  OR "population pharmacokinetic*"[Title/Abstract]
  OR "model-informed drug development"[Title/Abstract]
  OR "precision dosing"[Title/Abstract]
  OR "dose individualization"[Title/Abstract]
  OR "therapeutic drug monitoring"[Title/Abstract]
  OR PBPK[Title/Abstract]
  OR pharmacometrics[Title/Abstract]
)
"""

In [ ]:
ML_BLOCK = """
(
  "machine learning"[tiab] OR
  "artificial intelligence"[tiab] OR
  "deep learning"[tiab]
)
"""
PMX_BLOCK = """
(
  "pharmacometrics"[tiab] OR
  "model-based drug development"[tiab] OR
  "MIDD"[tiab]
)
"""


REVIEW_BLOCK = """
(
  review[Publication Type] OR
  tutorial[tiab] OR
  overview[tiab] OR
  perspective[tiab] OR
  "state of the art"[tiab] OR
  guideline [tiab] OR
  "white paper" [tiab] OR
  "practical guide"[tiab]
)
"""
HUMAN_BLOCK = "humans[MeSH Terms]"

REVIEW_PUBMED_QUERY_PMX = f"""
{ML_BLOCK}
AND
{PMX_BLOCK}
AND
{REVIEW_BLOCK}
AND
{HUMAN_BLOCK}
"""

REVIEW_PUBMED_QUERY_PMX = f"""
{ML_BLOCK}
AND
{PMX_BLOCK}
AND
{HUMAN_BLOCK}
"""


In [ ]:
pmids = get_pmids(
    REVIEW_PUBMED_QUERY_PMX,
    days_back=365*25,
    max_results=100
)
len(pmids)

In [ ]:
for pmid in pmids[:50]:
    article = query_pmid(pmid)
    print(f"- {article['title']}")


In [ ]:
print("Found paper:", any("Adoption of Machine Learning in Pharmacometrics" in query_pmid(p)["title"] for p in pmids))


In [ ]:
REVIEW_PUBMED_QUERY_PMX = f"""
("machine learning"[tiab] OR "artificial intelligence"[tiab] OR "deep learning"[tiab])
AND
("pharmacometrics"[tiab] OR "model-based drug development"[tiab] OR "MIDD"[tiab] 
     OR "population PK"[tiab] OR "population PD"[tiab])
"""


In [ ]:
pmids = get_pmids(
    REVIEW_PUBMED_QUERY_PMX,
    days_back=365*25,
    max_results=100
)
len(pmids)

In [ ]:
for pmid in pmids[:50]:
    article = query_pmid(pmid)
    print(f"- {article['title']}")
